# Databricks DBU, Cluster Utilization, and Job Usage Demo

This notebook provides four read-only reports:

1. **Cluster usage** ranked by DBU, with CPU, memory, runtime, and configured/observed worker counts.
2. **Job Cluster usage by Job ID**, with run count, average runtime, DBU per run, CPU, memory, and observed worker range.
3. **SQL Warehouse usage** ranked by DBU.
4. **SQL Warehouse configuration snapshot**, with Warehouse size, type, autoscaling range, and auto-stop configuration.

The notebook only calls `display()`. It does not persist data, resize compute, terminate compute, or modify any Databricks resource.

## Prerequisites

- A Unity Catalog-enabled workspace with the required System Tables enabled.
- `USE CATALOG` on `system`.
- `USE SCHEMA` and `SELECT` permissions for the tables used by this notebook:
  - `system.compute.clusters`
  - `system.compute.node_timeline`
  - `system.compute.warehouses`
  - `system.billing.usage`
  - `system.lakeflow.jobs`
  - `system.lakeflow.job_run_timeline`
  - `system.lakeflow.job_task_run_timeline`
- A cluster that can access Unity Catalog System Tables.
- For account-wide mode, a Databricks account service principal assigned to the target workspaces.

## Account Service Principal Setup

The service principal is required only for `ACCOUNT_ALL_WORKSPACES`. `CURRENT_WORKSPACE` uses the notebook's native Databricks identity and does not require these credentials.

### 1. Create the Databricks service principal

As a Databricks account admin:

1. Open the Databricks account console: https://accounts.azuredatabricks.net.
2. Select **User management** > **Service principals** > **Add service principal**.
3. Choose **Databricks managed** or **Microsoft Entra ID managed**.
4. For an Entra-managed principal, enter its Microsoft Entra application/client ID.
5. Give the principal a descriptive name such as `cluster-inventory-spn`.
6. Grant only the account permission required to list the workspaces in scope. For a proof of concept, Account Admin is the simplest option but is broader than necessary and should be reduced for production.

### 2. Assign the service principal to each target workspace

From the account console:

1. Select **Workspaces**.
2. Open a target workspace and select **Permissions**.
3. Select **Add permissions** and add the service principal.
4. Repeat for every workspace that the account-wide inventory should include.

Workspace assignment controls which workspaces the principal can access. Workspace Admin is convenient for a proof of concept but is not required by the SQL itself; use the least privilege supported by your operating model.

### 3. Generate a Databricks OAuth secret

1. In the account console, open **User management** > **Service principals**.
2. Select the service principal.
3. Open **Credentials & secrets** and generate an OAuth secret.
4. Record the client ID and secret immediately. The secret is displayed only once.
5. Set an expiration and rotation process. Databricks OAuth secrets can be valid for at most two years.

Use the **Databricks OAuth secret**, not a Microsoft Entra client secret, with `AccountClient`. Never place the secret directly in this notebook, source control, shell history, or logs.

### 4. Create the notebook secret scope

Create a Databricks-backed secret scope in the workspace where this notebook runs:

```bash
databricks secrets create-scope cluster-inventory
```

Add the three keys expected by the code. Run each command and enter the value interactively when prompted:

```bash
databricks secrets put-secret cluster-inventory account-id
databricks secrets put-secret cluster-inventory client-id
databricks secrets put-secret cluster-inventory client-secret
```

The values are:

- `account-id`: the Databricks account ID.
- `client-id`: the Databricks service principal application/client ID.
- `client-secret`: the Databricks OAuth secret generated in the preceding step.

### 5. Grant Secret Scope read access

The identity that executes this notebook must have `READ` on the scope:

```bash
databricks secrets put-acl cluster-inventory <notebook-run-principal> READ
```

Use the interactive user's email address when running the notebook manually, or the Run-as service principal application ID when running it as a scheduled Job.

### 6. Grant System Table access

The Account SPN is used by this notebook only to call `AccountClient.workspaces.list()`. The System Tables queries execute as the notebook's actual run identity. Grant that run identity the required Unity Catalog privileges:

```sql
GRANT USE CATALOG ON CATALOG system TO `<notebook-run-principal>`;

GRANT USE SCHEMA ON SCHEMA system.compute TO `<notebook-run-principal>`;
GRANT SELECT ON SCHEMA system.compute TO `<notebook-run-principal>`;

GRANT USE SCHEMA ON SCHEMA system.billing TO `<notebook-run-principal>`;
GRANT SELECT ON SCHEMA system.billing TO `<notebook-run-principal>`;

GRANT USE SCHEMA ON SCHEMA system.lakeflow TO `<notebook-run-principal>`;
GRANT SELECT ON SCHEMA system.lakeflow TO `<notebook-run-principal>`;
```

An administrator who is both an account admin and metastore admin must grant System Table access. System Tables are read-only.

## How to Run

1. Import this notebook into a Databricks workspace.
2. Attach it to an existing Unity Catalog-compatible cluster.
3. Run the installation cell once. Python restarts automatically after installing `databricks-sdk`.
4. Select the desired values in the notebook widgets:
   - `RUN_MODE`
   - `LOOKBACK_DAYS`
   - `TOP_N`
5. Run all remaining cells.
6. Review the four displayed result tables in order: Cluster Usage, Job Cluster Usage, SQL Warehouse DBU, and SQL Warehouse Configuration Snapshot.

## Parameters

### `RUN_MODE`

- `CURRENT_WORKSPACE` uses notebook-native authentication and filters all reports to the current workspace.
- `ACCOUNT_ALL_WORKSPACES` discovers workspaces with `AccountClient.workspaces.list()`. It reads `account-id`, `client-id`, and `client-secret` from the `cluster-inventory` secret scope.

Compute and Lakeflow System Tables are regional. Account-wide mode can discover workspace IDs across the account, but a notebook run returns compute and job records only for workspaces represented in the current workspace's cloud region. Run the notebook once per Databricks region for complete multi-region coverage.

### `LOOKBACK_DAYS`

Default: `30`. This controls the time window for DBU, node utilization, Job Run, and SQL Warehouse usage records. The Warehouse configuration snapshot always shows the latest active configuration and does not use this time window.

### `TOP_N`

Default: `50`. This limits the number of rows displayed by each report.

## Coverage Boundaries

- Classic Cluster CPU, memory, and workers come from `system.compute.node_timeline`.
- The Job report includes only runs whose compute resolves to a Cluster with `cluster_source = 'JOB'`. It excludes All-purpose Compute, Serverless Jobs, and SQL Warehouse tasks.
- Serverless compute has no `cluster_source` or Classic Cluster node telemetry. Report Serverless Jobs separately with `system.billing.usage` and `system.lakeflow` tables.
- SQL Warehouse `warehouse_size` is a Databricks capacity tier, not an Azure VM SKU.
- SQL Warehouse DBU is available from billing records, but SQL Warehouse CPU, memory, and worker-node metrics are not available from Classic Cluster node telemetry.
- Recent billing records can arrive later than compute telemetry, so DBU can temporarily be lower than expected.

## Setup References

- Manage service principals: https://learn.microsoft.com/azure/databricks/admin/users-groups/manage-service-principals
- OAuth M2M authentication: https://learn.microsoft.com/azure/databricks/dev-tools/auth/oauth-m2m
- Secret management: https://learn.microsoft.com/azure/databricks/security/secrets/
- System Tables access: https://learn.microsoft.com/azure/databricks/admin/system-tables/
- SQL Warehouse System Table: https://learn.microsoft.com/azure/databricks/admin/system-tables/warehouses

In [ ]:
%pip install "databricks-sdk>=0.81.0"
dbutils.library.restartPython()

In [ ]:
from urllib.parse import urlparse

from databricks.sdk import AccountClient, WorkspaceClient

CURRENT_WORKSPACE = "CURRENT_WORKSPACE"
ACCOUNT_ALL_WORKSPACES = "ACCOUNT_ALL_WORKSPACES"

def widget_value(name, default, label, choices=None):
    try:
        return dbutils.widgets.get(name)
    except Exception:
        if choices:
            dbutils.widgets.dropdown(name, default, choices, label)
        else:
            dbutils.widgets.text(name, default, label)
        return dbutils.widgets.get(name)


RUN_MODE = widget_value(
    "RUN_MODE",
    CURRENT_WORKSPACE,
    "Scope",
    [CURRENT_WORKSPACE, ACCOUNT_ALL_WORKSPACES],
).strip().upper()
LOOKBACK_DAYS = int(widget_value("LOOKBACK_DAYS", "30", "Lookback days"))
TOP_N = int(widget_value("TOP_N", "50", "Top rows"))

ACCOUNT_HOST = "https://accounts.azuredatabricks.net"
SECRET_SCOPE = "cluster-inventory"

if RUN_MODE == CURRENT_WORKSPACE:
    workspace_client = WorkspaceClient()
    current_host = workspace_client.config.host.rstrip("/")
    try:
        context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        workspace_ids = [str(context.workspaceId().get())]
    except Exception:
        workspace_ids = [urlparse(current_host).hostname.split(".")[0].removeprefix("adb-")]
else:
    account_id = dbutils.secrets.get(SECRET_SCOPE, "account-id")
    client_id = dbutils.secrets.get(SECRET_SCOPE, "client-id")
    client_secret = dbutils.secrets.get(SECRET_SCOPE, "client-secret")
    account_client = AccountClient(
        host=ACCOUNT_HOST,
        account_id=account_id,
        client_id=client_id,
        client_secret=client_secret,
    )
    workspace_ids = [
        str(workspace.workspace_id)
        for workspace in account_client.workspaces.list()
        if workspace.workspace_id
    ]

if not workspace_ids:
    raise RuntimeError("No workspaces are available for the selected mode.")

workspace_filter = ", ".join(f"'{value}'" for value in sorted(set(workspace_ids)))
print(f"RUN_MODE={RUN_MODE}; workspaces={len(set(workspace_ids))}; days={LOOKBACK_DAYS}; top={TOP_N}")

In [ ]:
def try_display(report_name, sql_text):
    try:
        frame = spark.sql(sql_text)
        print(report_name)
        display(frame)
        return frame
    except Exception as exc:
        print(f"{report_name} unavailable: {type(exc).__name__}: {exc}")
        return None

## Demo 1: Cluster Usage Ranked by DBU

### Data Sources

- `system.compute.clusters`: latest Cluster name, source, worker node type, and configured worker range.
- `system.compute.node_timeline`: minute-level CPU, memory, and active worker observations.
- `system.billing.usage`: DBU records attributed through `usage_metadata.cluster_id`.

### Collection Logic

1. Select the most recent configuration record for each `(workspace_id, cluster_id)` from `system.compute.clusters`.
2. Aggregate `node_timeline` to one row per Cluster per minute:
   - CPU = average of `cpu_user_percent + cpu_system_percent` across observed nodes.
   - Memory = average of `mem_used_percent` across observed nodes.
   - Active workers = distinct non-driver instances.
3. Aggregate minute records to the requested lookback window:
   - Runtime hours = observed Cluster minutes divided by 60.
   - Average and P95 CPU.
   - Average and P95 memory.
   - Minimum and maximum observed workers.
4. Sum DBU records by Cluster ID from `system.billing.usage`.
5. Union configuration, metric, and billing keys so that a Cluster is retained even if one source has no matching rows.
6. Sort by DBU descending, then runtime descending.

### How to Read the Result

- `configured_min_workers` and `configured_max_workers` are configuration limits.
- `observed_min_workers` and `observed_max_workers` are actual worker counts seen in minute telemetry.
- A Single Node Cluster correctly reports zero workers because the driver is not counted as a worker.
- Null utilization fields mean no matching node telemetry was available in the selected time window.

In [ ]:
cluster_sql = f"""
WITH latest_cluster AS (
  SELECT
    CAST(workspace_id AS STRING) AS workspace_id,
    cluster_id,
    cluster_name,
    cluster_source,
    worker_node_type,
    COALESCE(min_autoscale_workers, worker_count) AS configured_min_workers,
    COALESCE(max_autoscale_workers, worker_count) AS configured_max_workers
  FROM system.compute.clusters
  WHERE CAST(workspace_id AS STRING) IN ({workspace_filter})
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY workspace_id, cluster_id ORDER BY change_time DESC
  ) = 1
), per_minute AS (
  SELECT
    CAST(workspace_id AS STRING) AS workspace_id,
    cluster_id,
    DATE_TRUNC('minute', start_time) AS minute,
    AVG(cpu_user_percent + cpu_system_percent) AS avg_cpu_percent,
    AVG(mem_used_percent) AS avg_memory_percent,
    COUNT(DISTINCT CASE WHEN NOT driver THEN instance_id END) AS active_workers
  FROM system.compute.node_timeline
  WHERE start_time >= CURRENT_TIMESTAMP() - INTERVAL {LOOKBACK_DAYS} DAYS
    AND CAST(workspace_id AS STRING) IN ({workspace_filter})
  GROUP BY CAST(workspace_id AS STRING), cluster_id, DATE_TRUNC('minute', start_time)
), metrics AS (
  SELECT
    workspace_id,
    cluster_id,
    COUNT(*) / 60.0 AS runtime_hours,
    AVG(avg_cpu_percent) AS avg_cpu_percent,
    PERCENTILE_APPROX(avg_cpu_percent, 0.95) AS p95_cpu_percent,
    AVG(avg_memory_percent) AS avg_memory_percent,
    PERCENTILE_APPROX(avg_memory_percent, 0.95) AS p95_memory_percent,
    MIN(active_workers) AS observed_min_workers,
    MAX(active_workers) AS observed_max_workers
  FROM per_minute
  GROUP BY workspace_id, cluster_id
), billing AS (
  SELECT
    CAST(workspace_id AS STRING) AS workspace_id,
    usage_metadata.cluster_id AS cluster_id,
    SUM(usage_quantity) AS dbus
  FROM system.billing.usage
  WHERE usage_start_time >= CURRENT_TIMESTAMP() - INTERVAL {LOOKBACK_DAYS} DAYS
    AND CAST(workspace_id AS STRING) IN ({workspace_filter})
    AND usage_unit = 'DBU'
    AND usage_metadata.cluster_id IS NOT NULL
  GROUP BY CAST(workspace_id AS STRING), usage_metadata.cluster_id
), cluster_keys AS (
  SELECT workspace_id, cluster_id FROM latest_cluster
  UNION
  SELECT workspace_id, cluster_id FROM metrics
  UNION
  SELECT workspace_id, cluster_id FROM billing
)
SELECT
  k.workspace_id,
  k.cluster_id,
  c.cluster_name,
  c.cluster_source,
  c.worker_node_type,
  ROUND(COALESCE(b.dbus, 0), 2) AS dbus,
  ROUND(m.runtime_hours, 2) AS runtime_hours,
  ROUND(m.avg_cpu_percent, 2) AS avg_cpu_percent,
  ROUND(m.p95_cpu_percent, 2) AS p95_cpu_percent,
  ROUND(m.avg_memory_percent, 2) AS avg_memory_percent,
  ROUND(m.p95_memory_percent, 2) AS p95_memory_percent,
  c.configured_min_workers,
  c.configured_max_workers,
  m.observed_min_workers,
  m.observed_max_workers
FROM cluster_keys k
LEFT JOIN latest_cluster c USING (workspace_id, cluster_id)
LEFT JOIN metrics m USING (workspace_id, cluster_id)
LEFT JOIN billing b USING (workspace_id, cluster_id)
ORDER BY dbus DESC, runtime_hours DESC
LIMIT {TOP_N}
"""

cluster_usage_summary = try_display(
    "Cluster usage ranked by DBU",
    cluster_sql,
 )

## Demo 2: Job Cluster Usage by Job ID

### Data Sources

- `system.lakeflow.jobs`: latest Job name.
- `system.lakeflow.job_task_run_timeline`: Job ID, Job Run ID, run period, and `compute_ids`.
- `system.lakeflow.job_run_timeline`: user-visible run name.
- `system.compute.clusters`: resolves compute IDs and identifies Job Clusters through `cluster_source = 'JOB'`.
- `system.compute.node_timeline`: CPU, memory, and worker observations during each Job Run.
- `system.billing.usage`: DBU attributed by `usage_metadata.job_id` and `usage_metadata.job_run_id`.

### Collection Logic

1. Explode `job_task_run_timeline.compute_ids` to map each Job Run to its compute resources.
2. Join those IDs to the latest Cluster configuration and retain only `cluster_source = 'JOB'`. This excludes All-purpose Compute, Serverless Jobs, and SQL Warehouse tasks.
3. Build each Job Run window using the earliest task start and latest task end.
4. Sum billing records by `(workspace_id, job_id, job_run_id)`.
5. Read only node records whose Cluster ID belongs to that Job Run and whose timestamp is inside the Run window.
6. Aggregate node data per Job Run: average CPU, average memory, and minimum/maximum active workers.
7. Aggregate all eligible Runs by Job ID:
   - Run count.
   - Average elapsed runtime per Run.
   - Total DBU.
   - Average DBU per Run.
   - Average CPU and memory across Run-level observations.
   - Minimum and maximum workers observed across all Runs.

### How to Read the Result

- `avg_dbu_per_run` is the arithmetic average of DBU attributed to eligible Job Cluster Runs.
- `min_workers` and `max_workers` are observed worker counts, not configured autoscaling limits.
- CPU includes user and system CPU across the nodes observed during each Run.
- An empty table means no Run in the selected period resolved to a Cluster whose source was `JOB`.

In [ ]:
job_sql = f"""
WITH latest_job AS (
  SELECT
    CAST(workspace_id AS STRING) AS workspace_id,
    CAST(job_id AS STRING) AS job_id,
    name AS job_name
  FROM system.lakeflow.jobs
  WHERE CAST(workspace_id AS STRING) IN ({workspace_filter})
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY workspace_id, job_id ORDER BY change_time DESC
  ) = 1
), latest_cluster AS (
  SELECT
    CAST(workspace_id AS STRING) AS workspace_id,
    cluster_id,
    cluster_name
  FROM system.compute.clusters
  WHERE CAST(workspace_id AS STRING) IN ({workspace_filter})
    AND cluster_source = 'JOB'
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY workspace_id, cluster_id ORDER BY change_time DESC
  ) = 1
), task_compute AS (
  SELECT DISTINCT
    CAST(t.workspace_id AS STRING) AS workspace_id,
    CAST(t.job_id AS STRING) AS job_id,
    CAST(t.job_run_id AS STRING) AS job_run_id,
    compute_id AS cluster_id
  FROM system.lakeflow.job_task_run_timeline t
  LATERAL VIEW EXPLODE(t.compute_ids) exploded AS compute_id
  WHERE t.period_start_time >= CURRENT_TIMESTAMP() - INTERVAL {LOOKBACK_DAYS} DAYS
    AND CAST(t.workspace_id AS STRING) IN ({workspace_filter})
    AND t.job_run_id IS NOT NULL
), job_cluster_runs AS (
  SELECT
    t.workspace_id,
    t.job_id,
    t.job_run_id,
    t.cluster_id,
    c.cluster_name
  FROM task_compute t
  INNER JOIN latest_cluster c
    ON t.workspace_id = c.workspace_id
   AND t.cluster_id = c.cluster_id
), run_bounds AS (
  SELECT
    CAST(workspace_id AS STRING) AS workspace_id,
    CAST(job_id AS STRING) AS job_id,
    CAST(job_run_id AS STRING) AS job_run_id,
    MIN(period_start_time) AS run_start_time,
    MAX(period_end_time) AS run_end_time
  FROM system.lakeflow.job_task_run_timeline
  WHERE period_start_time >= CURRENT_TIMESTAMP() - INTERVAL {LOOKBACK_DAYS} DAYS
    AND CAST(workspace_id AS STRING) IN ({workspace_filter})
    AND job_run_id IS NOT NULL
  GROUP BY
    CAST(workspace_id AS STRING),
    CAST(job_id AS STRING),
    CAST(job_run_id AS STRING)
), run_names AS (
  SELECT
    CAST(workspace_id AS STRING) AS workspace_id,
    CAST(job_id AS STRING) AS job_id,
    CAST(run_id AS STRING) AS job_run_id,
    MAX(run_name) AS run_name
  FROM system.lakeflow.job_run_timeline
  WHERE period_start_time >= CURRENT_TIMESTAMP() - INTERVAL {LOOKBACK_DAYS} DAYS
    AND CAST(workspace_id AS STRING) IN ({workspace_filter})
  GROUP BY
    CAST(workspace_id AS STRING),
    CAST(job_id AS STRING),
    CAST(run_id AS STRING)
), eligible_runs AS (
  SELECT
    b.workspace_id,
    b.job_id,
    b.job_run_id,
    b.run_start_time,
    b.run_end_time,
    n.run_name,
    COLLECT_SET(c.cluster_id) AS cluster_ids,
    COLLECT_SET(c.cluster_name) AS cluster_names
  FROM run_bounds b
  INNER JOIN job_cluster_runs c
    ON b.workspace_id = c.workspace_id
   AND b.job_id = c.job_id
   AND b.job_run_id = c.job_run_id
  LEFT JOIN run_names n
    ON b.workspace_id = n.workspace_id
   AND b.job_id = n.job_id
   AND b.job_run_id = n.job_run_id
  GROUP BY
    b.workspace_id,
    b.job_id,
    b.job_run_id,
    b.run_start_time,
    b.run_end_time,
    n.run_name
), billing_per_run AS (
  SELECT
    CAST(workspace_id AS STRING) AS workspace_id,
    usage_metadata.job_id AS job_id,
    usage_metadata.job_run_id AS job_run_id,
    SUM(usage_quantity) AS dbus
  FROM system.billing.usage
  WHERE usage_start_time >= CURRENT_TIMESTAMP() - INTERVAL {LOOKBACK_DAYS} DAYS
    AND CAST(workspace_id AS STRING) IN ({workspace_filter})
    AND usage_unit = 'DBU'
    AND usage_metadata.job_id IS NOT NULL
    AND usage_metadata.job_run_id IS NOT NULL
  GROUP BY
    CAST(workspace_id AS STRING),
    usage_metadata.job_id,
    usage_metadata.job_run_id
), node_per_minute AS (
  SELECT
    r.workspace_id,
    r.job_id,
    r.job_run_id,
    DATE_TRUNC('minute', n.start_time) AS minute,
    AVG(n.cpu_user_percent + n.cpu_system_percent) AS avg_cpu_percent,
    AVG(n.mem_used_percent) AS avg_memory_percent,
    COUNT(DISTINCT CASE WHEN NOT n.driver THEN n.instance_id END) AS active_workers
  FROM eligible_runs r
  INNER JOIN system.compute.node_timeline n
    ON r.workspace_id = CAST(n.workspace_id AS STRING)
   AND ARRAY_CONTAINS(r.cluster_ids, n.cluster_id)
   AND n.start_time >= r.run_start_time
   AND n.start_time < r.run_end_time
  GROUP BY
    r.workspace_id,
    r.job_id,
    r.job_run_id,
    DATE_TRUNC('minute', n.start_time)
), metrics_per_run AS (
  SELECT
    workspace_id,
    job_id,
    job_run_id,
    AVG(avg_cpu_percent) AS avg_cpu_percent,
    AVG(avg_memory_percent) AS avg_memory_percent,
    MIN(active_workers) AS min_workers,
    MAX(active_workers) AS max_workers
  FROM node_per_minute
  GROUP BY workspace_id, job_id, job_run_id
), per_run AS (
  SELECT
    r.workspace_id,
    r.job_id,
    r.job_run_id,
    r.run_name,
    r.cluster_ids,
    r.cluster_names,
    GREATEST(
      UNIX_TIMESTAMP(r.run_end_time) - UNIX_TIMESTAMP(r.run_start_time),
      0
    ) / 3600.0 AS runtime_hours,
    b.dbus,
    m.avg_cpu_percent,
    m.avg_memory_percent,
    m.min_workers,
    m.max_workers
  FROM eligible_runs r
  LEFT JOIN billing_per_run b
    ON r.workspace_id = b.workspace_id
   AND r.job_id = b.job_id
   AND r.job_run_id = b.job_run_id
  LEFT JOIN metrics_per_run m
    ON r.workspace_id = m.workspace_id
   AND r.job_id = m.job_id
   AND r.job_run_id = m.job_run_id
), job_summary AS (
  SELECT
    workspace_id,
    job_id,
    MAX(run_name) AS observed_run_name,
    SORT_ARRAY(ARRAY_DISTINCT(FLATTEN(COLLECT_LIST(cluster_ids)))) AS cluster_ids,
    SORT_ARRAY(ARRAY_DISTINCT(FLATTEN(COLLECT_LIST(cluster_names)))) AS cluster_names,
    COUNT(*) AS run_count,
    AVG(runtime_hours) AS avg_runtime_hours,
    SUM(COALESCE(dbus, 0)) AS total_dbus,
    AVG(COALESCE(dbus, 0)) AS avg_dbus_per_run,
    AVG(avg_cpu_percent) AS avg_cpu_percent,
    AVG(avg_memory_percent) AS avg_memory_percent,
    MIN(min_workers) AS min_workers,
    MAX(max_workers) AS max_workers
  FROM per_run
  GROUP BY workspace_id, job_id
)
SELECT
  s.workspace_id,
  s.job_id,
  COALESCE(j.job_name, s.observed_run_name) AS job_name,
  ARRAY_JOIN(s.cluster_ids, ', ') AS job_cluster_ids,
  ARRAY_JOIN(s.cluster_names, ', ') AS job_cluster_names,
  s.run_count,
  ROUND(s.avg_runtime_hours, 2) AS avg_runtime_hours,
  ROUND(s.total_dbus, 2) AS total_dbus,
  ROUND(s.avg_dbus_per_run, 2) AS avg_dbus_per_run,
  ROUND(s.avg_cpu_percent, 2) AS avg_cpu_percent,
  ROUND(s.avg_memory_percent, 2) AS avg_memory_percent,
  s.min_workers,
  s.max_workers
FROM job_summary s
LEFT JOIN latest_job j USING (workspace_id, job_id)
ORDER BY total_dbus DESC, avg_runtime_hours DESC
LIMIT {TOP_N}
"""

job_usage_summary = try_display(
    "Job-cluster usage by job ID",
    job_sql,
 )

## Demo 3: SQL Warehouse Usage Ranked by DBU

### Data Source

- `system.billing.usage`: DBU attributed through `usage_metadata.warehouse_id`.

### Collection Logic

1. Filter billing records to the requested workspaces and lookback window.
2. Keep only records measured in `DBU` with a non-null Warehouse ID.
3. Sum DBU by `(workspace_id, warehouse_id)`.
4. Sort by DBU descending.

### How to Read the Result

- This report intentionally requires only billing access, so it can still work without `SELECT` on `system.compute.warehouses`.
- It returns Warehouse IDs and DBU only. Add `system.compute.warehouses` if Warehouse name, type, size, and scaling configuration are required.
- SQL Warehouse CPU, memory, and worker-node utilization are not joined to Classic Cluster node telemetry. Use Warehouse Monitoring and `system.query.history` for SQL performance analysis.

In [ ]:
warehouse_sql = f"""
SELECT
  CAST(workspace_id AS STRING) AS workspace_id,
  usage_metadata.warehouse_id AS warehouse_id,
  ROUND(SUM(usage_quantity), 2) AS dbus
FROM system.billing.usage
WHERE usage_start_time >= CURRENT_TIMESTAMP() - INTERVAL {LOOKBACK_DAYS} DAYS
  AND CAST(workspace_id AS STRING) IN ({workspace_filter})
  AND usage_unit = 'DBU'
  AND usage_metadata.warehouse_id IS NOT NULL
GROUP BY CAST(workspace_id AS STRING), usage_metadata.warehouse_id
ORDER BY dbus DESC
LIMIT {TOP_N}
"""

warehouse_usage_summary = try_display(
    "SQL Warehouse usage ranked by DBU",
    warehouse_sql,
 )

## Demo 4: SQL Warehouse Type, Size, and Autoscaling Snapshot

This report returns the latest active configuration snapshot for each SQL Warehouse, including the Warehouse type, size tier, scale-out range, and derived Classic/Pro worker-node capacity.

### Data Source

- `system.compute.warehouses`: slowly changing snapshots of SQL Warehouse configuration.

### Collection Logic

1. Filter Warehouse snapshots to the selected workspace IDs.
2. Rank snapshots by `change_time` within each `(workspace_id, warehouse_id)`.
3. Keep the most recent snapshot only.
4. Exclude Warehouses whose latest snapshot has a non-null `delete_time`.
5. Map the published Classic/Pro Warehouse size tiers to workers per cluster:
   - `2X_SMALL = 1`
   - `X_SMALL = 2`
   - `SMALL = 4`
   - `MEDIUM = 8`
   - `LARGE = 16`
   - `X_LARGE = 32`
   - `2X_LARGE = 64`
   - `3X_LARGE = 128`
   - `4X_LARGE = 256`
   - `5X_LARGE = 512`
6. Multiply workers per cluster by `min_clusters` and `max_clusters` to show the configured minimum and maximum worker-node capacity.

### How to Read the Result

- `warehouse_type` identifies `CLASSIC`, `PRO`, or `SERVERLESS`.
- `warehouse_size` is the Databricks SQL Warehouse size tier, such as `LARGE` or `X_LARGE`.
- `worker_nodes_per_cluster` is populated only for Classic and Pro Warehouses using the published Azure sizing table.
- `configured_min_worker_nodes` and `configured_max_worker_nodes` describe configured capacity boundaries, not currently running nodes.
- For Serverless Warehouses, worker-node fields are `NULL` because the infrastructure is managed dynamically by Databricks and is not exposed as a fixed node topology.
- `autoscaling_enabled` is `true` when `max_clusters` is greater than `min_clusters`.
- Use `system.compute.warehouse_events` for historical running-cluster counts and scale events.
- This report requires `SELECT` on `system.compute.warehouses`.

In [ ]:
warehouse_snapshot_sql = f"""
WITH ranked_warehouses AS (
  SELECT
    CAST(workspace_id AS STRING) AS workspace_id,
    warehouse_id,
    warehouse_name,
    warehouse_type,
    warehouse_channel,
    warehouse_size,
    min_clusters,
    max_clusters,
    auto_stop_minutes,
    change_time,
    delete_time,
    ROW_NUMBER() OVER (
      PARTITION BY workspace_id, warehouse_id
      ORDER BY change_time DESC
    ) AS row_number
  FROM system.compute.warehouses
  WHERE CAST(workspace_id AS STRING) IN ({workspace_filter})
), latest_active AS (
  SELECT *
  FROM ranked_warehouses
  WHERE row_number = 1
    AND delete_time IS NULL
), sized AS (
  SELECT
    *,
    CASE
      WHEN warehouse_type = 'SERVERLESS' THEN NULL
      WHEN warehouse_size = '2X_SMALL' THEN 1
      WHEN warehouse_size = 'X_SMALL' THEN 2
      WHEN warehouse_size = 'SMALL' THEN 4
      WHEN warehouse_size = 'MEDIUM' THEN 8
      WHEN warehouse_size = 'LARGE' THEN 16
      WHEN warehouse_size = 'X_LARGE' THEN 32
      WHEN warehouse_size = '2X_LARGE' THEN 64
      WHEN warehouse_size = '3X_LARGE' THEN 128
      WHEN warehouse_size = '4X_LARGE' THEN 256
      WHEN warehouse_size = '5X_LARGE' THEN 512
      ELSE NULL
    END AS worker_nodes_per_cluster
  FROM latest_active
)
SELECT
  workspace_id,
  warehouse_id,
  warehouse_name,
  warehouse_type,
  warehouse_channel,
  warehouse_size,
  worker_nodes_per_cluster,
  min_clusters,
  max_clusters,
  max_clusters > min_clusters AS autoscaling_enabled,
  worker_nodes_per_cluster * min_clusters AS configured_min_worker_nodes,
  worker_nodes_per_cluster * max_clusters AS configured_max_worker_nodes,
  auto_stop_minutes,
  configuration_change_time
FROM (
  SELECT
    *,
    change_time AS configuration_change_time
  FROM sized
)
ORDER BY workspace_id, warehouse_name
LIMIT {TOP_N}
"""

warehouse_configuration_snapshot = try_display(
    "SQL Warehouse type, size, and autoscaling snapshot",
    warehouse_snapshot_sql,
 )

## Validated Example Results

Validation environment: test workspace `2086896878562850`, `CURRENT_WORKSPACE` mode, 30-day lookback, validated on 2026-09-23. Parent Run ID: `548639744879669`. Values are time-sensitive and will change as telemetry and billing records arrive.

### Demo 1 Result: Cluster Usage

- 33 Cluster rows were returned.
- The highest-DBU Cluster was `0911-022942-ix20gj2w`.
- Sample metrics:
  - DBU: `1.43`
  - Runtime: `3.08` hours
  - Average CPU: `8.21%`
  - P95 CPU: `25.90%`
  - Average memory: `64.89%`
  - P95 memory: `77.16%`
  - Configured workers: `0-0`
  - Observed workers: `0-0`
- The zero worker count is expected because this is a Single Node Cluster; utilization still includes the driver node.

### Demo 2 Result: Job Cluster Usage

- The query completed successfully and returned the expected 13-column schema.
- The 30-day result contained zero rows because no Job Cluster Run existed in the validation data.

### Demo 3 Result: SQL Warehouse Usage

- One SQL Warehouse billing row was returned.
- Warehouse ID: `d87af32bf572d65c`.
- DBU: `4.29`.

### Demo 4 Result: SQL Warehouse Type, Size, and Autoscaling Snapshot

Two active Warehouse configuration rows were returned:

1. `Starter Warehouse`
   - Type: `PRO`
   - Size: `2X_SMALL`
   - Workers per cluster: `1`
   - Cluster range: `1-1`
   - Configured worker-node range: `1-1`
   - Autoscaling enabled: `false`
   - Auto-stop: `60` minutes
2. `servelesssql`
   - Type: `SERVERLESS`
   - Size: `2X_SMALL`
   - Cluster range: `1-1`
   - Autoscaling enabled: `false`
   - Auto-stop: `10` minutes
   - Worker-node fields: `NULL`, as expected for Databricks-managed Serverless infrastructure

All four reports completed without an `unavailable` error.

## Official References

- Jobs System Table reference: https://learn.microsoft.com/azure/databricks/admin/system-tables/jobs
- Compute System Table reference: https://learn.microsoft.com/azure/databricks/admin/system-tables/compute
- SQL Warehouse System Table reference: https://learn.microsoft.com/azure/databricks/admin/system-tables/warehouses
- Billable Usage System Table reference: https://learn.microsoft.com/azure/databricks/admin/system-tables/billing
- Monitor Job Costs and Performance: https://learn.microsoft.com/azure/databricks/admin/system-tables/jobs-cost